In [ ]:
from datasets import load_dataset
from difflib import SequenceMatcher
import re
import subprocess

In [ ]:
from retirever_v2 import main

In [ ]:
dataset = load_dataset("tianyang/repobench_python_v1.1", split="cross_file_first")

In [ ]:
from collections import defaultdict 

grouped = defaultdict(int)

for item in dataset:
    repo_name = item['repo_name']
    grouped[repo_name] += 1

In [ ]:
sorted_groups=sorted(grouped.items(), key=lambda x: x[1], reverse=True)

In [ ]:
sorted_groups[0]

In [ ]:
query_format="""
Given file_name: {file_name}
Fetch the most important depnedencies from the repo to complete the following code
{code}"""



In [ ]:
neo4j_config = {
        "url": "bolt://localhost:7687",
        "username": "neo4j",
        "password": "nutanix_neo4j",
    }

In [ ]:
repo_path='/home/ec2-user/RepoBench/Significant-Gravitas_autostandup'


In [ ]:
venv_python = "/home/ec2-user/ai-hpc-codegen-rag/graphrag_venv/bin/python3"
script_path = "/home/ec2-user/ai-hpc-codegen-rag/generate_graph_with_args.py"
# Build command with arguments
cmd = [
    venv_python,
    script_path,
    "--root-dir",
    repo_path,
    "--requirements-path",
    "",
    "--url",
    neo4j_config["url"],
    "--username",
    neo4j_config["username"],
    "--password",
    neo4j_config["password"],
]

# Run subprocess with timeout
result = subprocess.run(
    cmd, cwd=repo_path, capture_output=True, text=True, timeout=3000
)

In [ ]:
# from semantic_retriever import build_index_from_graph, CHROMA_CLIENT, COLLECTION_NAME, retrieve_from_index


In [ ]:
# print("Building semantic index...")
# try:
#     # Clear collection before building new index to avoid mixing repos
#     try:
        
#         CHROMA_CLIENT.delete_collection(name=COLLECTION_NAME)
#         print("Collection deleted")
#     except Exception:
#         pass  # Collection might not exist, which is fine
#     collection = CHROMA_CLIENT.get_or_create_collection(
#         name=COLLECTION_NAME
#     )

#     success = build_index_from_graph(repo_name, collection, neo4j_config)
#     if success:
#         print("  ✓ Semantic index built successfully.")
#     else:
#         print("  ✗ Semantic index building failed.")
# except Exception as e:
#     print(f"  ✗ Error building semantic index: {e}")


In [ ]:
examples=[item for item in dataset if item['repo_name'] == 'Significant-Gravitas/autostandup']

In [ ]:
examples[1]

In [ ]:
len(examples)

In [ ]:
print(examples[2]['context'][examples[2]['gold_snippet_index']])

In [ ]:
question=(query_format.format(file_name=examples[2]['file_path'].replace('/', '.').replace('.py', ''), code=examples[2]['cropped_code']))
print(question)

In [ ]:
res=main(question=question)


In [ ]:
res

### Evaluation

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
repos_to_avoid=['qitan/devops-backend-lite','tobagin/whakarere', 'StareAbyss/FoodsVsMouses_AutoAssistant', 'LkPrtctrd/BSL-V53','GXNU-ZhongLab/ODTrack',  'BrianPugh/cyclopts']

In [ ]:
with open('/home/ec2-user/ai-hpc-codegen-rag/repobench_accuracy_tracking_graph_unixcoder_2.json', 'r') as f:
    data = json.load(f)

accuracies = []
mrrs = []
ndcgs = []
total_correct_retrievals = 0
total_examples = 0

for repo, stats in data.items():
    if repo in repos_to_avoid:
        continue
    else:
        accuracies.append(stats['accuracy'])
        mrrs.append(stats['mrr'])
        ndcgs.append(stats['ndcg'])
        total_correct_retrievals += stats['correct_retrievals']
        total_examples += stats['total_examples']

# Calculate overall accuracy
if total_examples > 0:
    overall_accuracy = total_correct_retrievals / total_examples
    print(f"Overall Accuracy (excluding zero-accuracy repos): {overall_accuracy:.4f}")

    if mrrs:
        overall_mrr = sum(mrrs) / len(mrrs)
        print(f"Overall MRR (excluding zero-accuracy repos): {overall_mrr:.4f}")
    
    if ndcgs:
        overall_ndcg = sum(ndcgs) / len(ndcgs)
        print(f"Overall NDCG (excluding zero-accuracy repos): {overall_ndcg:.4f}")
else:
    print("No repositories with non-zero accuracy found.")

# Print repos with zero accuracy


if not accuracies:
    print("\nNo data to plot for histogram as all repos have 0 accuracy.")


In [ ]:
with open('/home/ec2-user/ai-hpc-codegen-rag/repobench_accuracy_tracking_graph_qwen3_8b_2.json', 'r') as f:
    data = json.load(f)

accuracies = []
mrrs = []
ndcgs = []
total_correct_retrievals = 0
total_examples = 0
repo_count=0
for repo, stats in data.items():
    if repo in repos_to_avoid:
        continue
    else:
        accuracies.append(stats['accuracy'])
        mrrs.append(stats['mrr'])
        ndcgs.append(stats['ndcg'])
        total_correct_retrievals += stats['correct_retrievals']
        total_examples += stats['total_examples']
        repo_count += 1

print(f"Total repositories processed: {repo_count}")

# Calculate overall accuracy
if total_examples > 0:
    overall_accuracy = total_correct_retrievals / total_examples
    print(f"Overall Accuracy (excluding zero-accuracy repos): {overall_accuracy:.4f}")

    if mrrs:
        overall_mrr = sum(mrrs) / len(mrrs)
        print(f"Overall MRR (excluding zero-accuracy repos): {overall_mrr:.4f}")
    
    if ndcgs:
        overall_ndcg = sum(ndcgs) / len(ndcgs)
        print(f"Overall NDCG (excluding zero-accuracy repos): {overall_ndcg:.4f}")
else:
    print("No repositories with non-zero accuracy found.")

# Print repos with zero accuracy


if not accuracies:
    print("\nNo data to plot for histogram as all repos have 0 accuracy.")

In [ ]:
repo_count

In [ ]:
total_examples

In [ ]:
mean_accuracy = np.mean(accuracies)
median_accuracy = np.median(accuracies)

print(f"\nMean Accuracy: {mean_accuracy:.4f}")
print(f"Median Accuracy: {median_accuracy:.4f}")

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(accuracies, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(mean_accuracy, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_accuracy:.2f}')
plt.axvline(median_accuracy, color='green', linestyle='dashed', linewidth=2, label=f'Median: {median_accuracy:.2f}')

plt.title('Histogram of Repository Accuracies (excluding zeros)')
plt.xlabel('Accuracy')
plt.ylabel('Number of Repositories')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
with open('/home/ec2-user/ai-hpc-codegen-rag/repobench_accuracy_tracking_file_level_qwen3_8b.json', 'r') as f:
    data = json.load(f)

accuracies = []
mrrs = []
ndcgs = []
total_correct_retrievals = 0
total_examples = 0

for repo, stats in data.items():
    if repo in repos_to_avoid:
        continue
    else:
        accuracies.append(stats['accuracy'])
        mrrs.append(stats['mrr'])
        ndcgs.append(stats['ndcg'])
        total_correct_retrievals += stats['correct_retrievals']
        total_examples += stats['total_examples']

# Calculate overall accuracy
if total_examples > 0:
    overall_accuracy = total_correct_retrievals / total_examples
    print(f"Overall Accuracy (excluding zero-accuracy repos): {overall_accuracy:.4f}")

    if mrrs:
        overall_mrr = sum(mrrs) / len(mrrs)
        print(f"Overall MRR (excluding zero-accuracy repos): {overall_mrr:.4f}")
    
    if ndcgs:
        overall_ndcg = sum(ndcgs) / len(ndcgs)
        print(f"Overall NDCG (excluding zero-accuracy repos): {overall_ndcg:.4f}")
else:
    print("No repositories with non-zero accuracy found.")

# Print repos with zero accuracy


if not accuracies:
    print("\nNo data to plot for histogram as all repos have 0 accuracy.")

In [ ]:
from find_imports_treesitter import find_imports, categorize_imports, load_requirements
from find_imports_treesitter import resolve_relative_import

In [ ]:
res=find_imports('/home/ec2-user/RepoBench/radekd91_inferno/inferno/models/video_emorec/VideoEmotionClassifier.py','')

In [ ]:
res